In [1029]:
import pandas as pd
import numpy as np
import requests
import camelot
from scipy.optimize import brentq

In [3]:
url = "https://apisidra.ibge.gov.br/values/t/1737/n1/all/v/2266/p/all/d/v2266%2013"

dados = requests.get(url).json()

ipca_indice = pd.DataFrame(dados, )


In [841]:
df_raw = pd.DataFrame(dados)

# Primeira linha explica o que cada código significa
header = df_raw.iloc[0]

# Dados verdadeiros começam da linha 1 em diante
df = df_raw.iloc[1:].copy()

# Descobre automaticamente as colunas corretas
col_data_codigo = header[header == "Mês (Código)"].index[0]
col_mes = header[header == "Mês"].index[0]
col_valor = header[header == "Valor"].index[0]

ipca_indice = pd.DataFrame({
    "data_codigo": df[col_data_codigo],
    "mes": df[col_mes],
    "numero_indice": df[col_valor]
})

ipca_indice["data"] = pd.to_datetime(ipca_indice["data_codigo"], format="%Y%m")

ipca_indice["numero_indice"] = (
    ipca_indice["numero_indice"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

ipca_indice = ipca_indice[["data", "mes", "numero_indice"]]

In [843]:
ipca_indice = ipca_indice[ipca_indice['data'] > '2024-02-01']

In [845]:
ipca_indice["razao_ni"] = (
    ipca_indice["numero_indice"] /
    ipca_indice["numero_indice"].shift(1)
)

In [847]:
pdf = './CRI Hortis - Termo de Securitização - TCMB 29052024 Versão Final.docx.pdf'

tabelas = camelot.read_pdf(
    pdf,
    pages="99-103",
    flavor="lattice"
)

In [849]:
colunas = [
    "#",
    "data_aniversario",
    "data_pagamento",
    "juros",
    "incorporacao_juros",
    "amortizacao",
    "percentual_amortizado"
]

dfs = []

for i, tabela in enumerate(tabelas):
    df = tabela.df.copy()

    # limpa quebras de linha e espaços
    df = df.map(lambda x: x.replace("\n", " ").strip() if isinstance(x, str) else x)

    # garante que a tabela tem 6 colunas
    if df.shape[1] != 7:
        print(f"Pulando tabela {i}: tem {df.shape[1]} colunas")
        continue

    # na primeira tabela, remove a linha de cabeçalho
    if i == 0:
        df = df.iloc[1:].reset_index(drop=True)

    # força as mesmas colunas em todas as tabelas
    df.columns = colunas

    # guarda origem, opcional
    df["tabela_origem"] = i

    dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)

In [851]:
df_final = df_final.drop(columns=['#', 'tabela_origem'])

df_final['data_aniversario'] = df_final['data_aniversario'].str.split(' ').apply(lambda x: x[-1])

In [853]:
df_final["data_aniversario"] = pd.to_datetime(
    df_final["data_aniversario"],
    dayfirst=True,
    errors="coerce"
)

df_final["data_pagamento"] = pd.to_datetime(
    df_final["data_pagamento"],
    dayfirst=True,
    errors="coerce"
)

In [855]:
df_final["proxima_data"] = df_final["data_aniversario"].shift(-1)

df_final['dcp'] = (df_final["proxima_data"] - df_final["data_aniversario"]).dt.days
df_final['dct'] = df_final['dcp']
df_final['dcp'] = df_final['dcp'] - 1
df_final['dct'] = df_final['dct'] - 1

df_final['dcp'].iloc[0] = 17
df_final['dct'].iloc[0] = 31

C:\Users\edson\AppData\Local\Temp\ipykernel_17436\2328867896.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_final['dcp'].iloc[0] = 17
C:\Users\edson\AppData\Local\Temp\ipykernel_17436\2328867896.py:8: SettingWithCopyWarning: 
A value

In [857]:
df_final['percentual_amortizado'] = df_final['percentual_amortizado'].str.split('%').str[0]

In [859]:
df_final['percentual_amortizado'] = df_final['percentual_amortizado'].str.replace(',', '.').astype('float')

In [861]:
df_final['percentual_amortizado'] = df_final['percentual_amortizado'] / 100

In [863]:
df_final['percentual_amortizado'] = pd.to_numeric(df_final['percentual_amortizado'], errors="coerce")

In [865]:
ipca_indice

,data,mes,numero_indice,razao_ni
532,2024-03-01,março 2024,6869.14,NaN
533,2024-04-01,abril 2024,6895.24,1.003800
534,2024-05-01,maio 2024,6926.96,1.004600
535,2024-06-01,junho 2024,6941.51,1.002100
536,2024-07-01,julho 2024,6967.89,1.003800
537,2024-08-01,agosto 2024,6966.50,0.999801
538,2024-09-01,setembro 2024,6997.15,1.004400
539,2024-10-01,outubro 2024,7036.33,1.005599
540,2024-11-01,novembro 2024,7063.77,1.003900
541,2024-12-01,dezembro 2024,7100.50,1.005200


In [867]:
df_final["data_aniversario"] = df_final["data_aniversario"].dt.to_period("M")
ipca_indice["data"] = ipca_indice["data"].dt.to_period("M")

In [1090]:
ipca_indice

,data,mes,numero_indice,razao_ni
532,2024-03,março 2024,6869.14,NaN
533,2024-04,abril 2024,6895.24,1.003800
534,2024-05,maio 2024,6926.96,1.004600
535,2024-06,junho 2024,6941.51,1.002100
536,2024-07,julho 2024,6967.89,1.003800
537,2024-08,agosto 2024,6966.50,0.999801
538,2024-09,setembro 2024,6997.15,1.004400
539,2024-10,outubro 2024,7036.33,1.005599
540,2024-11,novembro 2024,7063.77,1.003900
541,2024-12,dezembro 2024,7100.50,1.005200


In [869]:
df_final["mes_razao_usada"] = df_final["data_aniversario"] - 2

In [1092]:
df_final = df_final.merge(
    ipca_indice[["data", "razao_ni"]],
    left_on="mes_razao_usada",
    right_on="data",
    how="left",
    suffixes=("", "razao_ni")
)

df_final = df_final.drop(columns=['data', 'proxima_data'])

KeyError: "['proxima_data'] not found in axis"

In [873]:
df_final["razao_ni"] = df_final["razao_ni"].clip(lower=1)

In [875]:
df_final["fator_ipca"] = np.where(
    df_final["razao_ni"].notna(),
    df_final["razao_ni"] ** (df_final["dcp"] / df_final["dct"]),
    np.nan
)

In [877]:
df_final["fator_ipca"].cumprod()

0      1.002082
1      1.006692
2      1.008806
3      1.012640
4      1.012640
         ...   
200         NaN
201         NaN
202         NaN
203         NaN
204         NaN
Name: fator_ipca, Length: 205, dtype: float64

In [879]:
df_final['fator_juros'] = (1 + 9 / 100) ** (df_final["dcp"] / 365)

In [881]:
df_final['produtorio_juros'] = df_final["fator_juros"].cumprod()

In [893]:
df_final

,data_aniversario,data_pagamento,juros,incorporacao_juros,amortizacao,percentual_amortizado,dcp,dct,mes_razao_usada,razao_ni,fator_ipca,fator_juros,produtorio_juros,amortizacao_extraordinaria
0,2024-06,2024-06-18,Não,Sim,Não,0.000000,17.0,31.0,2024-04,1.0038,1.002082,1.004022,1.004022,0.00
1,2024-07,2024-07-16,Sim,Não,Sim,0.002193,30.0,30.0,2024-05,1.0046,1.004600,1.007108,1.011159,2.50
2,2024-08,2024-08-16,Sim,Não,Sim,0.002149,30.0,30.0,2024-06,1.0021,1.002100,1.007108,1.018346,5.52
3,2024-09,2024-09-17,Sim,Não,Sim,0.002169,29.0,29.0,2024-07,1.0038,1.003800,1.006870,1.025343,3.57
4,2024-10,2024-10-16,Sim,Não,Sim,0.002255,30.0,30.0,2024-08,1.0000,1.000000,1.007108,1.032631,3.57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,2041-02,2041-02-18,Sim,Não,Sim,0.197083,27.0,27.0,2040-12,NaN,NaN,1.006395,4.030755,0.00
201,2041-03,2041-03-18,Sim,Não,Sim,0.247526,30.0,30.0,2041-01,NaN,NaN,1.007108,4.059407,0.00
202,2041-04,2041-04-16,Sim,Não,Sim,0.330897,29.0,29.0,2041-02,NaN,NaN,1.006870,4.087297,0.00
203,2041-05,2041-05-16,Sim,Não,Sim,0.498229,30.0,30.0,2041-03,NaN,NaN,1.007108,4.116350,0.00


In [887]:
df_amort_extr = pd.read_csv('./amort_extraord.csv')

In [889]:
df_amort_extr['amortizacao_extraordinaria'] = df_amort_extr['amortizacao_extraordinaria'].str.replace(',', '.').astype('float')

In [891]:
df_final['amortizacao_extraordinaria'] = df_amort_extr['amortizacao_extraordinaria']

In [979]:
df_final['fator_ipca'] = df_final['fator_ipca'].fillna(1)

In [981]:
# seu df já pronto
df = df_final.copy()

# valor inicial que você quer colocar na primeira linha
x_inicial = 1002.87

# cria as colunas que serão calculadas
df["vna_inicio"] = 0.0
df["vna_antes_pagamento"] = 0.0
df["juros"] = 0.0
df["vna_depois_pagamento"] = 0.0
df["amortizacao_reais"] = 0.0
df["total_amortizado"] = 0.0

for i in range(1, len(df)):

    if i == 1:
        df.loc[i, "vna_inicio"] = x_inicial
        df.loc[i, "total_amortizado"] = 0
    else:
        df.loc[i, "vna_inicio"] = df.loc[i - 1, "vna_depois_pagamento"]
        df.loc[i, "total_amortizado"] = df.loc[i - 1, "total_amortizado"]

    df.loc[i, "vna_corrigido"] = (
        df.loc[i, "vna_inicio"]
        * df.loc[i, "fator_ipca"]
    )

    df.loc[i, "juros"] = (
        df.loc[i, "vna_corrigido"]
        * (df.loc[i, "fator_juros"] - 1)
    )

    df.loc[i, "vna_antes_pagamento"] = (
        df.loc[i, "vna_corrigido"]
        + df.loc[i, "juros"]
    )

    # percentual amortizado sobre o VNA antes do pagamento
    df.loc[i, "amortizacao_reais"] = (
        df.loc[i, "vna_corrigido"]
        * df.loc[i, "percentual_amortizado"]
    )

    # total amortizado acumulado até essa linha
    df.loc[i, "total_amortizado"] = (
        df.loc[i, "total_amortizado"]
        + df.loc[i, "amortizacao_reais"]
    )

    df.loc[i, "vna_depois_pagamento"] = (
        df.loc[i, "vna_corrigido"]
        - df.loc[i, "amortizacao_reais"]
        - df.loc[i, "amortizacao_extraordinaria"]
    )

In [997]:
df

,data_aniversario,data_pagamento,juros,incorporacao_juros,amortizacao,percentual_amortizado,dcp,dct,mes_razao_usada,razao_ni,fator_ipca,fator_juros,produtorio_juros,amortizacao_extraordinaria,vna_inicio,vna_antes_pagamento,vna_depois_pagamento,amortizacao_reais,total_amortizado,vna_corrigido
0,2024-06,2024-06-18,0.000000,Sim,Não,0.000000,17.0,31.0,2024-04,1.0038,1.002082,1.004022,1.004022,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
1,2024-07,2024-07-16,7.161437,Não,Sim,0.002193,30.0,30.0,2024-05,1.0046,1.004600,1.007108,1.011159,2.50,1002.870000,1014.644915,1002.774066,2.209411,2.209411,1007.483478
2,2024-08,2024-08-16,7.142934,Não,Sim,0.002149,30.0,30.0,2024-06,1.0021,1.002100,1.007108,1.018346,5.52,1002.774066,1012.023316,997.200894,2.159488,4.368899,1004.880382
3,2024-09,2024-09-17,6.877295,Não,Sim,0.002169,29.0,29.0,2024-07,1.0038,1.003800,1.006870,1.025343,3.57,997.200894,1007.867877,995.249434,2.171149,6.540048,1000.990582
4,2024-10,2024-10-16,7.074475,Não,Sim,0.002255,30.0,30.0,2024-08,1.0000,1.000000,1.007108,1.032631,3.57,995.249434,1002.323908,989.435146,2.244287,8.784335,995.249434
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,2041-02,2041-02-18,0.316299,Não,Sim,0.197083,27.0,27.0,2040-12,NaN,1.000000,1.006395,4.030755,0.00,49.459213,49.775512,39.711643,9.747570,1035.598710,49.459213
201,2041-03,2041-03-18,0.282280,Não,Sim,0.247526,30.0,30.0,2041-01,NaN,1.000000,1.007108,4.059407,0.00,39.711643,39.993923,29.881979,9.829664,1045.428375,39.711643
202,2041-04,2041-04-16,0.205304,Não,Sim,0.330897,29.0,29.0,2041-02,NaN,1.000000,1.006870,4.087297,0.00,29.881979,30.087282,19.994122,9.887857,1055.316232,29.881979
203,2041-05,2041-05-16,0.142123,Não,Sim,0.498229,30.0,30.0,2041-03,NaN,1.000000,1.007108,4.116350,0.00,19.994122,20.136245,10.032470,9.961651,1065.277883,19.994122


In [1007]:
df_fluxo = pd.DataFrame()

df_fluxo['data'] = df['data_aniversario']
df_fluxo['juros'] = df['juros']
df_fluxo['amortizacao'] = df['amortizacao_extraordinaria'] + df['amortizacao_reais']

In [1021]:
df_fluxo["fluxo"] = (
    df_fluxo["juros"].fillna(0)
    + df_fluxo["amortizacao"].fillna(0)
)

df_fluxo["data"] = df_fluxo["data"].dt.to_timestamp() + pd.DateOffset(days=14)

In [1023]:
df_fluxo

,data,juros,amortizacao,fluxo
0,2024-06-15,0.000000,0.000000,0.000000
1,2024-07-15,7.161437,4.709411,11.870848
2,2024-08-15,7.142934,7.679488,14.822422
3,2024-09-15,6.877295,5.741149,12.618444
4,2024-10-15,7.074475,5.814287,12.888762
...,...,...,...,...
200,2041-02-15,0.316299,9.747570,10.063869
201,2041-03-15,0.282280,9.829664,10.111944
202,2041-04-15,0.205304,9.887857,10.093161
203,2041-05-15,0.142123,9.961651,10.103774


In [1025]:
def calcular_pu(df, taxa_desconto, data_base, coluna_data="data", coluna_fluxo="fluxo"):
    df_calc = df.copy()

    df_calc[coluna_data] = pd.to_datetime(df_calc[coluna_data])
    data_base = pd.to_datetime(data_base)

    # pega apenas fluxos futuros
    df_calc = df_calc[df_calc[coluna_data] > data_base].copy()

    df_calc["dias"] = (df_calc[coluna_data] - data_base).dt.days

    df_calc["fator_desconto"] = (1 + taxa_desconto) ** (df_calc["dias"] / 365)

    df_calc["vp_fluxo"] = df_calc[coluna_fluxo] / df_calc["fator_desconto"]

    pu = df_calc["vp_fluxo"].sum()

    return pu

In [1027]:
calcular_pu(
    df=df_fluxo,
    taxa_desconto=0.09,
    data_base="2026-06-24"
)


1001.8821945744413

In [1035]:
def calcular_taxa_implicita(
    df,
    pu_alvo,
    data_base,
    coluna_data="data",
    coluna_fluxo="fluxo",
    taxa_min=-0.99,
    taxa_max=1.00
):
    def erro(taxa):
        pu_calculado = calcular_pu(
            df=df,
            taxa_desconto=taxa,
            data_base=data_base,
            coluna_data=coluna_data,
            coluna_fluxo=coluna_fluxo
        )
        return pu_calculado - pu_alvo

    taxa = brentq(erro, taxa_min, taxa_max)

    return taxa

In [1043]:
calcular_taxa_implicita(
    df=df_fluxo,
    pu_alvo=1100,
    data_base="2026-06-24"
)


0.07345126788312592

In [1047]:
data_base = pd.to_datetime("2026-06-22")
taxa_desconto = 0.10  # 10% a.a.

# pega apenas fluxos futuros
df_pu = df_fluxo[df_fluxo["data"] > data_base].copy()

# garante fluxo sem NaN
df_pu["fluxo"] = df_pu["fluxo"].fillna(0)

# dias corridos até cada pagamento
df_pu["dias_corridos"] = (
    df_pu["data"] - data_base
).dt.days

# fator de desconto em dias corridos
df_pu["fator_desconto"] = (
    1 + taxa_desconto
) ** (df_pu["dias_corridos"] / 365)

# valor presente do fluxo
df_pu["vp_fluxo"] = (
    df_pu["fluxo"] / df_pu["fator_desconto"]
)

# PU na data-base
pu = df_pu["vp_fluxo"].sum()

In [1051]:
df_pu

,data,juros,amortizacao,fluxo,dias_corridos,fator_desconto,vp_fluxo
25,2026-07-15,7.228057,2.803471,10.031528,23,1.006024,9.971461
26,2026-08-15,7.208129,2.752138,9.960267,54,1.014201,9.820806
27,2026-09-15,6.948126,2.771974,9.720100,85,1.022444,9.506734
28,2026-10-15,7.168863,2.864220,10.033083,115,1.030485,9.736276
29,2026-11-15,6.909403,2.812842,9.722245,146,1.038860,9.358570
...,...,...,...,...,...,...,...
200,2041-02-15,0.316299,9.747570,10.063869,5352,4.045214,2.487846
201,2041-03-15,0.282280,9.829664,10.111944,5380,4.074899,2.481520
202,2041-04-15,0.205304,9.887857,10.093161,5411,4.108018,2.456941
203,2041-05-15,0.142123,9.961651,10.103774,5441,4.140326,2.440333


In [1053]:
pu

948.8831471437168